## 1) Install (if required)

In [ ]:
# !pip install huggingface-hub

## 2) HuggingFace Login

In [ ]:
from huggingface_hub import login
from google.colab import userdata
hf_token = userdata.get('huggingface_login') # Fetching the Hugging Face token from the Colab user data
login(token = hf_token) # Logging into Hugging Face Hub to access models and other resources

## 3) Download model at the desired path

In [ ]:
from huggingface_hub import snapshot_download

In [ ]:
model_id="google/gemma-2-2b-it"  # Replace with the ID of the model you want to download
snapshot_download(repo_id=model_id, local_dir="gemma-2-2b")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/29.1k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

'/content/gemma-2-2b'

## 4) Clone llama.cpp from GitHub

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 42970, done.
remote: Counting objects: 100% (7823/7823), done.
remote: Compressing objects: 100% (476/476), done.
remote: Total 42970 (delta 7621), reused 7347 (delta 7347), pack-reused 35147 (from 2)
Receiving objects: 100% (42970/42970), 77.16 MiB | 15.99 MiB/s, done.
Resolving deltas: 100% (31519/31519), done.


## 5) Install Libraries required for llama.cpp

In [ ]:
!pip install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.8/186.8 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.2/76.2 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: torch
    Found existing installation: torch 2.5.1+cu124
    Uninstalling torch-2.5.1+cu124:
      Successfully uninstalled torch-2.5.1+cu124
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.20.1+cu124 requires torch==2.5.1, but you have torch 2.2.2+cpu which is incompatible.
torchaudio 2.5.1+cu124 requires torch==2.5.1, but you have torch 2.2.2+cpu which is incompatible.


## 6) Command to convert .safetensors model to .gguf

!python {path to convert_hf_to_gguf.py} {path to hf_model} --outfile {name_of_outputfile.gguf} --outtype {quantization type}


In [ ]:
!python "/content/llama.cpp/convert_hf_to_gguf.py" "/content/gemma-2-2b" --outfile "gemma-2-2b.gguf" --outtype q8_0

INFO:hf-to-gguf:Loading model: gemma-2-2b
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:token_embd.weight,                 torch.bfloat16 --> Q8_0, shape = {2304, 256000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.bfloat16 --> F32, shape = {2304}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.bfloat16 --> Q8_0, shape = {9216, 2304}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.bfloat16 --> Q8_0, shape = {2304, 9216}
INFO:hf-to-gguf:blk.0.ffn_up.weight,               torch.bfloat16 --> Q8_0, shape = {2304, 9216}
INFO:hf-to-gguf:blk.0.post_attention_norm.weight,  torch.bfloat16 --> F32, shape = {2304}
INFO:hf-to-gguf:blk.0.post_ffw_norm.weight,        torch.bfloat16 --> F32, shape = {2304}
INFO:hf-to-gguf:blk.0.ffn_norm.weig

## 7) Calculating Size Reduction after GGUF conversion

In [ ]:
import os  # Importing the os module for file operations

def get_file_size(path):
    """Returns the size of a file in megabytes (MB)."""
    size_bytes = os.path.getsize(path)  # Get file size in bytes
    size_mb = (size_bytes / 1024) / 1024  # Convert bytes to megabytes
    return size_mb

# Define paths for the original model's parts
original_model_path1 = "/content/gemma-2-2b/model-00001-of-00002.safetensors"
original_model_path2 = "/content/gemma-2-2b/model-00002-of-00002.safetensors"

# Calculate the total size of the original model by summing both parts
original_size = get_file_size(original_model_path1) + get_file_size(original_model_path2)

# Define the path for the quantized model
quantized_model_path = "gemma-2-2b.gguf"

# Calculate the size of the quantized model
quantized_size = get_file_size(quantized_model_path)

In [ ]:
print(f"Original Model Size: {original_size:.2f} MB")
print(f"Quantized Model Size: {quantized_size:.2f} MB")
print(f"Size Reduction: {((original_size - quantized_size) / original_size) * 100:.2f}%")

Original Model Size: 4986.49 MB
Quantized Model Size: 2655.50 MB
Size Reduction: 46.75%
